In [19]:
from z3 import *
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad

In [2]:
vals = [int(x) for x in open("predictable.txt") if x.strip()]
len(vals), vals[:8], vals[-8:]

N = 624
M = 397

A32    = BitVecVal(0x9908B0DF, 32)
UPPER  = BitVecVal(0x80000000, 32)
LOWER  = BitVecVal(0x7fffffff, 32)
ZERO32 = BitVecVal(0, 32)
ONE32  = BitVecVal(1, 32)

B32 = BitVecVal(0x9D2C5680, 32)
C32 = BitVecVal(0xEFC60000, 32)

def temper_mt(x):
    y = x ^ LShR(x, 11)
    y = y ^ ((y << 7) & B32)
    y = y ^ ((y << 15) & C32)
    y = y ^ LShR(y, 18)
    return y
def mt_next(xk, xk1, xk397):
    y = (xk & UPPER) | (xk1 & LOWER)
    xa = LShR(y, 1) ^ If((y & ONE32) == ONE32, A32, ZERO32)
    return xk397 ^ xa

In [3]:
def build_mt_solver(leaks, byte_idx=3, predict=4):
    total = len(leaks) + predict
    x = [BitVec(f"x_{i}", 32) for i in range(total)]

    s = Solver()
    s.set("timeout", 30000)

    for k in range(total - N):
        s.add(x[k + N] == mt_next(x[k], x[k + 1], x[k + M]))

    lo = 8 * byte_idx
    hi = lo + 7
    for i, leak in enumerate(leaks):
        t = temper_mt(x[i])
        s.add(Extract(hi, lo, t) == BitVecVal(leak, 8))

    return s, x

In [ ]:
prefixes = [624, 800, 1000, 1248, 1500, 1872, 2000, 2200, 2400, 2496, 2500]
for byte_idx, label in [
    (0, "LSB"),
    (1, "byte1"),
    (2, "byte2"),
    (3, "MSB"),
]:
    print(f"=== {label} ===")
    for n in prefixes:
        s, x = build_mt_solver(vals[:n], byte_idx=byte_idx, predict=0)
        res = s.check()
        print(f"n={n} -> {res}")
        if res == unsat:
            break
    print()

n=1000 -> sat


In [8]:
N = 624
M = 397
A = 0x9908B0DF

def xor_words(a, b):
    return [x ^ y for x, y in zip(a, b)]

def shr(word, amount):
    if amount == 0:
        return word[:]
    return word[amount:] + [0] * amount

def shl(word, amount):
    if amount == 0:
        return word[:]
    return [0] * amount + word[:-amount]

def and_mask(word, mask):
    return [bit if (mask >> i) & 1 else 0 for i, bit in enumerate(word)]

def temper(word):
    y = word[:]
    y = xor_words(y, shr(y, 11))
    y = xor_words(y, and_mask(shl(y, 7), 0x9D2C5680))
    y = xor_words(y, and_mask(shl(y, 15), 0xEFC60000))
    y = xor_words(y, shr(y, 18))
    return y

def twist(state):
    new_state = []
    for i in range(N):
        y = state[(i + 1) % N][:31] + [state[i][31]]
        xa = shr(y, 1)
        lsb = y[0]
        if lsb:
            for bit in range(32):
                if (A >> bit) & 1:
                    xa[bit] ^= lsb
        new_state.append(xor_words(state[(i + M) % N], xa))
    return new_state

def build_symbolic_state():
    state = []
    for word_index in range(N):
        word = []
        for bit_index in range(32):
            word.append(1 << (word_index * 32 + bit_index))
        state.append(word)
    return state

def add_equation(basis, coeff, rhs):
    while coeff:
        lead = coeff.bit_length() - 1
        existing = basis.get(lead)
        if existing is None:
            basis[lead] = (coeff, rhs)
            return
        coeff ^= existing[0]
        rhs ^= existing[1]
    if rhs:
        raise ValueError("inconsistent")

In [13]:
def check_mt_linear(vals, byte_start=0, start_index=0, limit=None, verbose=False):
    basis = {}
    state = build_symbolic_state()
    idx = start_index
    use_vals = vals if limit is None else vals[:limit]

    try:
        for out_i, leak in enumerate(use_vals):
            if idx >= N:
                state = twist(state)
                idx = 0

            y = temper(state[idx])

            for b in range(8):
                coeff = y[byte_start + b]
                rhs = (leak >> b) & 1
                add_equation(basis, coeff, rhs)

            idx += 1

            if verbose and out_i % 100 == 0:
                print(f"out_i={out_i}, rank={len(basis)}")

        return {
            "ok": True,
            "rank": len(basis),
            "bad_at": None,
            "used": len(use_vals),
        }

    except ValueError:
        return {
            "ok": False,
            "rank": len(basis),
            "bad_at": out_i,
            "used": len(use_vals),
        }


for start, label in [
    (0, "LSB"),
    (8, "byte1"),
    (16, "byte2"),
    (24, "MSB"),
]:
    # print(label, check_mt_linear(vals, byte_start=start, start_index=0, limit=1872))
    # print("byte1", check_mt_linear(vals, byte_start=8,  start_index=0, limit=2000))
    # print("MSB  ", check_mt_linear(vals, byte_start=24, start_index=0, limit=2000))
    print("MSB 2496", check_mt_linear(vals, byte_start=24, start_index=0, limit=2496))
    print("MSB 2500", check_mt_linear(vals, byte_start=24, start_index=0, limit=2500))

MSB 2496 {'ok': True, 'rank': 19967, 'bad_at': None, 'used': 2496}
MSB 2500 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}
MSB 2496 {'ok': True, 'rank': 19967, 'bad_at': None, 'used': 2496}
MSB 2500 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}
MSB 2496 {'ok': True, 'rank': 19967, 'bad_at': None, 'used': 2496}
MSB 2500 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}
MSB 2496 {'ok': True, 'rank': 19967, 'bad_at': None, 'used': 2496}
MSB 2500 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}


In [14]:
for s in [0, 1, 2, 619, 620, 621, 622, 623]:
      print("start_index =", s, check_mt_linear(vals, byte_start=24, start_index=s, limit=2500))

start_index = 0 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}
start_index = 1 {'ok': False, 'rank': 19960, 'bad_at': 2495, 'used': 2500}
start_index = 2 {'ok': False, 'rank': 19953, 'bad_at': 2494, 'used': 2500}
start_index = 619 {'ok': False, 'rank': 19967, 'bad_at': 2495, 'used': 2500}
start_index = 620 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}
start_index = 621 {'ok': False, 'rank': 19966, 'bad_at': 2495, 'used': 2500}
start_index = 622 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}
start_index = 623 {'ok': False, 'rank': 19968, 'bad_at': 2496, 'used': 2500}


In [15]:
def solve_basis(basis):
    solution = 0
    for lead in sorted(basis.keys(), reverse=True):
        coeff, rhs = basis[lead]
        parity = (coeff & solution).bit_count() & 1
        if parity ^ rhs:
            solution |= 1 << lead
    return solution

def eval_word(word, solution):
    value = 0
    for bit_index, coeff in enumerate(word):
        if (coeff & solution).bit_count() & 1:
            value |= 1 << bit_index
    return value

def predict_from_prefix(vals, limit=2496, count=4, byte_start=24, start_index=0):
    basis = {}
    state = build_symbolic_state()
    idx = start_index
    future = []

    for out_i in range(limit + count):
        if idx >= N:
            state = twist(state)
            idx = 0

        y = temper(state[idx])

        if out_i < limit:
            leak = vals[out_i]
            for b in range(8):
                add_equation(basis, y[byte_start + b], (leak >> b) & 1)
        else:
          future.append(y)

        idx += 1

    solution = solve_basis(basis)
    outputs = [eval_word(word, solution) for word in future]
    return basis, outputs

In [16]:
basis, outs = predict_from_prefix(vals, limit=2496, count=4, byte_start=24, start_index=0)
print("rank =", len(basis))
print("predicted 32-bit outputs =", [hex(x) for x in outs])
print("predicted top bytes =", [(x >> 24) & 0xff for x in outs])
print("actual last 4 leaked bytes =", vals[2496:2500])

rank = 19967
predicted 32-bit outputs = ['0xb73e2741', '0xe1ceeb92', '0x6e6a67cb', '0xc119eec7']
predicted top bytes = [183, 225, 110, 193]
actual last 4 leaked bytes = [37, 33, 244, 122]


In [17]:
def recover_prefix_model(vals, limit=2496, count=4, byte_start=24, start_index=0):
    basis = {}
    state = build_symbolic_state()
    idx = start_index
    future = []

    for out_i in range(limit + count):
        if idx >= N:
            state = twist(state)
            idx = 0

        y = temper(state[idx])

        if out_i < limit:
            leak = vals[out_i]
            for b in range(8):
                add_equation(basis, y[byte_start + b], (leak >> b) & 1)
        else:
            future.append(y)

        idx += 1

    return basis, future

def solve_homogeneous_with_freebit(basis, freebit):
    sol = 1 << freebit
    for lead in sorted(basis.keys(), reverse=True):
        coeff, _ = basis[lead]
        if (coeff & sol).bit_count() & 1:
            sol |= 1 << lead
    return sol

In [18]:
basis, future = recover_prefix_model(vals, limit=2496, count=4, byte_start=24, start_index=0)

sol0 = solve_basis(basis)

all_bits = set(range(624 * 32))
pivot_bits = set(basis.keys())
free_bits = sorted(all_bits - pivot_bits)

print("num free bits =", len(free_bits))
print("first free bit =", free_bits[:5])

outs0 = [eval_word(w, sol0) for w in future]
print("outs from particular solution =", [hex(x) for x in outs0])

if free_bits:
    ker = solve_homogeneous_with_freebit(basis, free_bits[0])
    outs1 = [eval_word(w, sol0 ^ ker) for w in future]
    print("outs after toggling free bit =", [hex(x) for x in outs1])
    print("same outputs?", outs0 == outs1)

num free bits = 1
first free bit = [0]
outs from particular solution = ['0xb73e2741', '0xe1ceeb92', '0x6e6a67cb', '0xc119eec7']
outs after toggling free bit = ['0xb73e2741', '0xe1ceeb92', '0x6e6a67cb', '0xc119eec7']
same outputs? True


In [21]:
ct = bytes.fromhex(
    "38c9e92b118434a05e9bffd3605602061268d420ab9bda37849b6fc99ff85c815"
    "64a6a4476b6144ccea81714e243eca0b32db8cdc0a40afb37cb810c48a16e80"
)

outs = [0xb73e2741, 0xe1ceeb92, 0x6e6a67cb, 0xc119eec7]

key = b"".join(x.to_bytes(4, "little") for x in outs)
print("key =", key.hex())

pt = AES.new(key, AES.MODE_ECB).decrypt(ct)
print("raw pt =", pt)

try:
    print("unpadded =", unpad(pt, 16))
except Exception as e:
    print("unpad error:", e)

key = 41273eb792ebcee1cb676a6ec7ee19c1
raw pt = b'{5\xdc\x86\xe8\xb9\xf8\x19\'9\xd2U\xbc\xc6\xa3l\x1b\xc9\xe2L\x81\x874\x0eZs\x9d\x19o\xeb^\xca\x940\xe4,%\x99ty\x83\xe5\xd9\xc8\xe2d\xc27\x97\xed\x8aCL0\xa5\xda\x1ab|\xac8"\x98\x00'
unpad error: Padding is incorrect.


In [22]:
from itertools import product
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad

ct = bytes.fromhex(
    "38c9e92b118434a05e9bffd3605602061268d420ab9bda37849b6fc99ff85c815"
    "64a6a4476b6144ccea81714e243eca0b32db8cdc0a40afb37cb810c48a16e80"
)

def try_key_layouts(outs):
    for order_name, seq in [
        ("forward", outs),
        ("reverse", outs[::-1]),
    ]:
        for endian in ["little", "big"]:
            key = b"".join(x.to_bytes(4, endian) for x in seq)
            pt = AES.new(key, AES.MODE_ECB).decrypt(ct)

            printable = sum(32 <= b < 127 or b in (9, 10, 13) for b in pt)
            print(f"order={order_name:7} endian={endian:6} printable={printable}/64 first16={pt[:16].hex()}")

            try:
                up = unpad(pt, 16)
                print("  unpadded =", up)
            except Exception:
                pass

In [23]:
try_key_layouts(outs)

order=forward endian=little printable=26/64 first16=7b35dc86e8b9f8192739d255bcc6a36c
order=forward endian=big    printable=26/64 first16=4ea15216f4ee22e77875ed2c64605738
order=reverse endian=little printable=23/64 first16=123a74bf619c05651e01acfcfb2f7dd3
order=reverse endian=big    printable=27/64 first16=b60ccc43aa720843f6d1ba819972acc5


In [24]:
def recover_candidate_outputs(vals, start_index, limit=2496, count=4, byte_start=24):
    basis, future = recover_prefix_model(
        vals,
        limit=limit,
        count=count,
        byte_start=byte_start,
        start_index=start_index,
    )
    sol = solve_basis(basis)
    outs = [eval_word(w, sol) for w in future]
    return basis, outs

for s in [0, 620, 622, 623]:
    basis, cand_outs = recover_candidate_outputs(vals, start_index=s, limit=2496, count=4, byte_start=24)
    print(f"start_index={s}, rank={len(basis)}, outs={[hex(x) for x in cand_outs]}")
    try_key_layouts(cand_outs)
    print("-" * 60)

start_index=0, rank=19967, outs=['0xb73e2741', '0xe1ceeb92', '0x6e6a67cb', '0xc119eec7']
order=forward endian=little printable=26/64 first16=7b35dc86e8b9f8192739d255bcc6a36c
order=forward endian=big    printable=26/64 first16=4ea15216f4ee22e77875ed2c64605738
order=reverse endian=little printable=23/64 first16=123a74bf619c05651e01acfcfb2f7dd3
order=reverse endian=big    printable=27/64 first16=b60ccc43aa720843f6d1ba819972acc5
------------------------------------------------------------
start_index=620, rank=19968, outs=['0xd7dac1da', '0x4744941', '0x750a96c4', '0x90efa468']
order=forward endian=little printable=21/64 first16=d7289d76f3457216a497b97a5dcccd47
order=forward endian=big    printable=26/64 first16=6b7d9acd5acf4e414a66355512a14af4
order=reverse endian=little printable=27/64 first16=256b5a0f70f01d4cc005332240920579
order=reverse endian=big    printable=28/64 first16=24c94c0f9953906b1764bd61ca93e7e9
------------------------------------------------------------
start_index=622, ra

In [25]:
def get_byte_lane(words, lane):
    return [((w >> (8 * lane)) & 0xff) for w in words]

target = vals[2496:2500]
print("target tail bytes =", target)

for s in [0, 620, 622, 623]:
    basis, cand_outs = recover_candidate_outputs(vals, start_index=s, limit=2496, count=4, byte_start=24)
    print(f"\nstart_index={s}, outs={[hex(x) for x in cand_outs]}")
    for lane, label in [
        (0, "LSB"),
        (1, "byte1"),
        (2, "byte2"),
        (3, "MSB"),
    ]:
        lane_bytes = get_byte_lane(cand_outs, lane)
        print(f"  {label:5} -> {lane_bytes} | match? {lane_bytes == target}")

target tail bytes = [37, 33, 244, 122]

start_index=0, outs=['0xb73e2741', '0xe1ceeb92', '0x6e6a67cb', '0xc119eec7']
  LSB   -> [65, 146, 203, 199] | match? False
  byte1 -> [39, 235, 103, 238] | match? False
  byte2 -> [62, 206, 106, 25] | match? False
  MSB   -> [183, 225, 110, 193] | match? False

start_index=620, outs=['0xd7dac1da', '0x4744941', '0x750a96c4', '0x90efa468']
  LSB   -> [218, 65, 196, 104] | match? False
  byte1 -> [193, 73, 150, 164] | match? False
  byte2 -> [218, 116, 10, 239] | match? False
  MSB   -> [215, 4, 117, 144] | match? False

start_index=622, outs=['0xe36ae6', '0x69e3c2f', '0xe2f0617b', '0x42ce4831']
  LSB   -> [230, 47, 123, 49] | match? False
  byte1 -> [106, 60, 97, 72] | match? False
  byte2 -> [227, 158, 240, 206] | match? False
  MSB   -> [0, 6, 226, 66] | match? False

start_index=623, outs=['0xc7f3f76', '0x254fd28c', '0xd7fddf3', '0x7cdc6a07']
  LSB   -> [118, 140, 243, 7] | match? False
  byte1 -> [63, 210, 221, 106] | match? False
  byte2 -> [1

In [26]:
def berlekamp_massey(bits):
    n = len(bits)
    c = [0] * n
    b = [0] * n
    c[0] = 1
    b[0] = 1
    L = 0
    m = -1

    for N in range(n):
        d = bits[N]
        for i in range(1, L + 1):
            d ^= c[i] & bits[N - i]

        if d == 1:
            t = c[:]
            shift = N - m
            for j in range(shift, n):
                c[j] ^= b[j - shift]
            if L <= N // 2:
                L = N + 1 - L
                m = N
                b = t
    return L

for bit in range(8):
    seq = [(x >> bit) & 1 for x in vals]
    print(f"bit {bit}: linear complexity = {berlekamp_massey(seq)}")


bit 0: linear complexity = 1249
bit 1: linear complexity = 1250
bit 2: linear complexity = 1250
bit 3: linear complexity = 1251
bit 4: linear complexity = 1250
bit 5: linear complexity = 1249
bit 6: linear complexity = 1251
bit 7: linear complexity = 1249
